### Bagging

Parallel ensembles: independent learners trained on different bootstrap samples, combined by equal vote/average, unlike boosting's sequential error-correction (see `boosting.ipynb`). Covers Random Forest (the standard bagging algorithm) and Stacking/Blending (a related but distinct combination strategy, learns how to combine rather than using a fixed rule).

## Bagging: Random Forest

#### 0. Core idea

Bagging, not boosting. Many trees trained in parallel, independently. Averaged at the end. Opposite of gradient boosting's sequential error-correction.

Toy setup used throughout: 2 features, mentions_IRS (0/1) and urgency_language (0/1). 4 docs:
```
doc1: mentions_IRS=1, urgency=1, label=gov_impersonation
doc2: mentions_IRS=1, urgency=0, label=gov_impersonation
doc3: mentions_IRS=0, urgency=1, label=romance_scam
doc4: mentions_IRS=0, urgency=0, label=romance_scam
```


#### 1. Bootstrap sampling

Each tree gets its own training set, sampled WITH replacement, same size as original. "Bagging" = Bootstrap AGGregatING.

From [doc1, doc2, doc3, doc4]:
```
tree_A sample: [doc1, doc1, doc3, doc4]   (doc2 left out, doc1 repeated)
tree_B sample: [doc2, doc3, doc3, doc4]   (different draw, different tree)
```
Roughly 63% of unique rows land in any given bootstrap sample on average (probability a specific row is NEVER drawn in n draws from n rows is (1-1/n)^n, approaches 1/e ≈ 0.368 as n grows, so ~63% chance it IS drawn at least once). The left-out ~37% per tree is called out-of-bag data, usable as a free internal validation set.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
docs = ["doc1", "doc2", "doc3", "doc4"]

def bootstrap_sample(data, rng):
    idx = rng.integers(0, len(data), size=len(data))
    return [data[i] for i in idx]

for i in range(3):
    print(f"tree_{i} sample:", bootstrap_sample(docs, rng))

#### 2. Random feature subsets

At each split, only a random subset of features is considered, not all of them. Default: sqrt(n_features) for classification.

Why. Without this, every tree keeps splitting on the single strongest feature first, trees end up nearly identical, averaging them buys nothing. Forcing each split to pick from a random subset decorrelates the trees, so averaging actually reduces variance.

With 2 features here, sqrt(2) rounds to 1: each split only gets to look at ONE randomly chosen feature (mentions_IRS OR urgency_language, not both), even though both exist.


#### 3. Splits: Gini impurity, not Gain

No gradients, no hessians. Splits found by Gini impurity (or entropy).

Formula: Gini = 1 - sum(p_k^2), p_k = fraction of class k in the node. Lower is purer.

Worked example, root node, all 4 docs (2 gov, 2 romance):
```
p_gov = 0.5, p_romance = 0.5
Gini_root = 1 - (0.5^2 + 0.5^2) = 1 - 0.5 = 0.5
```
Split on mentions_IRS:
```
left (IRS=0) = {doc3, doc4}, both romance -> Gini_left = 1 - (0^2 + 1^2) = 0
right (IRS=1) = {doc1, doc2}, both gov -> Gini_right = 1 - (1^2 + 0^2) = 0
weighted Gini after split = (2/4)(0) + (2/4)(0) = 0
Gini reduction = 0.5 - 0 = 0.5
```
Same feature that gave XGBoost its highest Gain also gives Random Forest its best Gini reduction here. Not a coincidence, mentions_IRS genuinely separates the two classes perfectly in this toy.


In [ ]:
def gini(labels):
    labels = np.array(labels)
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p ** 2)

root = ["gov", "gov", "romance", "romance"]
left = ["romance", "romance"]   # IRS=0
right = ["gov", "gov"]          # IRS=1

gini_root = gini(root)
weighted_gini = (len(left) / len(root)) * gini(left) + (len(right) / len(root)) * gini(right)

print("Gini root:", gini_root)
print("Weighted Gini after split:", weighted_gini)
print("Gini reduction:", gini_root - weighted_gini)

#### 4. Majority vote

Final prediction = majority vote across all trees. Equal, unweighted voice per tree. Not a weighted sum like boosting's F = sum(eta * tree_output).

5 trees predict for a new doc: [gov, gov, romance, gov, gov] -> 4 votes gov, 1 vote romance -> prediction = gov.

#### 5. Why averaging works: variance reduction

Variance of an average of N independent estimators shrinks by ~1/N. Trees here aren't fully independent (same data pool, overlapping features considered), so the reduction is less than 1/N but still real.

Bagging mainly reduces VARIANCE (averaging cancels out each tree's individual noise). Boosting mainly reduces BIAS (each new tree directly targets the current error). Different failure mode each is built to fix.

#### 6. Practical notes

Forgiving of default hyperparameters. Adding more trees almost never hurts (independent trees, errors don't compound), unlike boosting where too many rounds overfits. Good baseline / sanity-check model before reaching for XGBoost.

Underfitting: too few trees, max_depth too shallow, max_features too small relative to signal.
Overfitting: individual trees grown too deep with no min_samples_leaf floor (bagging reduces variance across trees, but a single tree grown fully deep still memorizes its own bootstrap sample).


In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = np.array([[1, 1], [1, 0], [0, 1], [0, 0]])  # mentions_IRS, urgency_language
y = ["gov", "gov", "romance", "romance"]

rf = RandomForestClassifier(n_estimators=5, max_features=1, random_state=42)
rf.fit(X, y)

print("prediction for [1, 1]:", rf.predict([[1, 1]]))
print("prediction for [0, 0]:", rf.predict([[0, 0]]))

# vote breakdown across the 5 trees for a new example
new_doc = [[1, 0]]
votes = [tree.predict(new_doc)[0] for tree in rf.estimators_]
print("individual tree votes for [1, 0]:", votes)

## Stacking / Blending

A third combination strategy, alongside bagging (parallel, equal vote) and boosting (sequential, error-correcting). Stacking trains several DIFFERENT base models, then trains a meta-model whose job is to learn how to best combine their predictions, rather than using a fixed rule like majority vote or a weighted sum.

#### 0. Core idea

Toy setup: two different base models (say logreg and a decision stump) each produce a predicted probability for 4 examples:
```
true labels:        [1,    0,    1,    0   ]
model A (logreg):    [0.90, 0.20, 0.60, 0.40]
model B (stump):     [0.60, 0.30, 0.55, 0.50]
```
A meta-model (often just another logreg) is trained using [pred_A, pred_B] as its two input features and the true labels as target, it learns, from data, how much to trust each base model, rather than someone hand-picking a fixed weighting like 0.5/0.5.

#### 1. The leakage trap, and why it needs out-of-fold predictions

If the base models' predictions used to train the meta-model come from predicting on the SAME data they were trained on, the meta-model sees artificially confident, overfit predictions, the base models already memorized answers for those exact rows. The meta-model would learn to trust base models more than it should, then fail on genuinely new data.

Fix: generate the meta-model's training features using out-of-fold predictions, same idea as k-fold cross-validation. Split training data into k folds, for each fold, train the base models on the other k-1 folds and predict on the held-out fold, stitch these held-out predictions together as the meta-model's training input. Every prediction used to train the meta-model comes from a base model that never saw that row during its own training, same leakage-avoidance principle as CatBoost's Ordered Boosting (see `boosting.ipynb`) and the cross-validation notebook, applied here to model combination instead of feature encoding or hyperparameter selection.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_predict

rng = np.random.default_rng(0)
X = rng.normal(size=(40, 3))
y = (X[:, 0] + X[:, 1] > 0).astype(int)

base_models = [
    ("logreg", LogisticRegression()),
    ("stump", DecisionTreeClassifier(max_depth=1)),
]

# sklearn's StackingClassifier handles the out-of-fold mechanics internally
stack = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(), cv=5)
stack.fit(X, y)
print("stacked prediction accuracy on training data:", stack.score(X, y))

# demonstrating the out-of-fold mechanic directly: this is what feeds the meta-model internally
oof_preds_logreg = cross_val_predict(LogisticRegression(), X, y, cv=5, method="predict_proba")[:, 1]
print("sample of out-of-fold predictions (never trained on their own row):", oof_preds_logreg[:5].round(3))

#### 2. Practical notes

Works best when base models make DIFFERENT kinds of mistakes, a linear model and a tree model tend to be more complementary than two very similar tree models, since the meta-model can learn to lean on whichever base model handles a given region of the data better. Cost: more models to train and maintain, and the extra cross-validation step for out-of-fold predictions, real added complexity for what is often only a small accuracy gain over the single best base model. Common in competition settings (Kaggle) where squeezing out the last bit of accuracy matters more than simplicity, less common as a default in production systems where a single well-tuned model (often one of the boosting variants) is usually preferred for maintainability.